# MAE Supplementary Figure 6 analysis

# 1. Load supplementary MAE figure inputs and configure paths

## General parameters

In [ ]:
from pathlib import Path
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Documented private input and output path variables
analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

ROOT = resolve_analysis_path(
    os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai"))
)
OUT_DIR = resolve_analysis_path(
    os.environ.get("MAE_FIGS6_OUTPUT_DIR", os.path.join("outputs", "ai", "FigS6"))
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
DISEASE_ORDER = ["CAR-T_CRS", "COVID19", "SLE"]
K90_BY_DISEASE = {"CAR-T_CRS": 229, "COVID19": 649, "SLE": 116}

# 2. Helper functions

In [ ]:
# ============================================================
# Fig. S6 — visualization and robustness companion to Fig. 6
#
# This notebook uses SAVED OUTPUTS ONLY.
#
# No:
#   - model loading
#   - H5AD loading
#   - encoder execution
#   - Integrated Gradients
#   - reversion reruns
#
# Planned panels:
#   S6A  Five-seed latent neighborhood stability
#   S6B  Matched-control calibration of program reversion
#   S6C  Seed-wise K90 stability
#   S6D  Disease-specific K90 gene expression profiles
#   S6E  Quadrant scaffold PDF
# ============================================================

from pathlib import Path
import os
import re
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

from matplotlib.lines import Line2D

from scipy.optimize import curve_fit


# 1. Paths
ROOT = resolve_analysis_path(
    os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai"))
)

OUT_DIR = (
    ROOT
    / "FigS6"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# 2. Display order
DISEASE_ORDER = [
    "CAR-T_CRS",
    "COVID19",
    "SLE",
]

DISEASE_LABELS = {
    "CAR-T_CRS": "CAR-T CRS",
    "COVID19": "COVID-19",
    "SLE": "SLE",
}


PROGRAM_ORDER = [
    "Inflammatory_CD14_IL1B",
    "Emergency_CD14_S100A8",
    "CD16_sensing_LST1",
]

PROGRAM_LABELS = {
    "Inflammatory_CD14_IL1B":
        "Inflammatory CD14\n(IL1B)",

    "Emergency_CD14_S100A8":
        "Emergency CD14\n(S100A8)",

    "CD16_sensing_LST1":
        "CD16 sensing\n(LST1)",
}


# Keep these consistent with the main figure
DISEASE_COLORS = {
    "CAR-T_CRS": "#4C78A8",
    "COVID19": "#F58518",
    "SLE": "#54A24B",
}

PROGRAM_COLORS = {
    "Inflammatory_CD14_IL1B": "#4C78A8",
    "Emergency_CD14_S100A8": "#E39C65",
    "CD16_sensing_LST1": "#55408D",
}

# 3. General plotting parameters
plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,

    # Preserve editable text
    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    # Prevent SVG text from being converted to paths
    "svg.fonttype": "none",
})


# 4. Small utilities
def require_file(path):
    """
    Stop immediately if an expected saved result is missing.

    This notebook is deliberately visualization-only, so it
    should never silently regenerate upstream analyses.
    """

    path = Path(path)

    if not path.exists():

        raise FileNotFoundError(
            f"Required saved result was not found:\n{path}"
        )

    return path


def newest_file(pattern, root=ROOT):
    """
    Find the most recently modified file matching a recursive
    glob pattern.

    Useful because the K90 analysis was rerun after the
    original August-12 workspace manifest.
    """

    matches = list(
        root.rglob(pattern)
    )

    if len(matches) == 0:

        raise FileNotFoundError(
            f"No file matching:\n{pattern}\n"
            f"under:\n{root}"
        )

    matches = sorted(
        matches,
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    print(
        f"{pattern}\n"
        f"  -> using {matches[0]}"
    )

    return matches[0]


def save_panel(fig, stem):
    """
    Save:
      PDF = vector-first figure assembly
      PNG = quick visual inspection

    dpi affects only rasterized artists in the PDF and the PNG.
    """

    pdf = OUT_DIR / f"{stem}.pdf"
    png = OUT_DIR / f"{stem}.png"

    fig.savefig(
        pdf,
        bbox_inches="tight",
    )

    fig.savefig(
        png,
        dpi=600,
        bbox_inches="tight",
    )

    print(
        f"Saved:\n"
        f"  {pdf}\n"
        f"  {png}"
    )


def first_existing_column(df, candidates):
    """
    Return the first candidate column that exists.
    """

    for col in candidates:

        if col in df.columns:
            return col

    return None


print(
    f"Output directory:\n{OUT_DIR}"
)

# 3. Fig. S6A latent neighborhood stability

In [ ]:
# ============================================================
# Fig. S6A
# Latent neighborhood stability across MAE seeds
# ============================================================

from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

# Paths
ROOT = resolve_analysis_path(
    os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai"))
)

OUT_DIR = ROOT / "FigS6"

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


STABILITY_FILE = (
    ROOT
    / "Fig6_MAE_seed_neighborhood_stability.csv"
)

# Load
stab = pd.read_csv(
    STABILITY_FILE
)


print(
    "Neighborhood-stability columns:"
)

print(
    stab.columns.tolist()
)

display(
    stab
)


# ------------------------------------------------------------
# Explicitly normalize seed IDs
#
# Important:
# iterrows() can upcast integer seed IDs to floats because the
# other columns in each row are floating-point values
#
# Converting to integer HERE avoids labels such as
# "20260810.0"
# ------------------------------------------------------------

stab["seed_a"] = (
    pd.to_numeric(
        stab["seed_a"]
    )
    .astype(int)
)

stab["seed_b"] = (
    pd.to_numeric(
        stab["seed_b"]
    )
    .astype(int)
)


SEEDS = sorted(
    set(
        stab["seed_a"]
    )
    |
    set(
        stab["seed_b"]
    )
)


print(
    "\nSeeds:",
    SEEDS,
)

# Construct symmetric pairwise matrix
# Use mean_kNN_overlap as the displayed stability metric
matrix = pd.DataFrame(
    np.nan,
    index=SEEDS,
    columns=SEEDS,
    dtype=float,
)


for row in stab.itertuples(
    index=False
):

    s1 = int(
        row.seed_a
    )

    s2 = int(
        row.seed_b
    )

    value = float(
        row.mean_knn_overlap
    )


    matrix.loc[
        s1,
        s2
    ] = value

    matrix.loc[
        s2,
        s1
    ] = value

# Self-neighborhood overlap is 1 by definition
np.fill_diagonal(
    matrix.values,
    1.0,
)

print(
    "\nPairwise mean kNN overlap:"
)

display(
    matrix
)

matrix.to_csv(
    OUT_DIR
    / "FigS6A_seed_neighborhood_stability_matrix.csv"
)

# Plot
fig, ax = plt.subplots(
    figsize=(4.0, 3.5)
)

im = ax.imshow(
    matrix.values,
    vmin=0,
    vmax=1,
    cmap="Blues",
)

seed_labels = [
    str(seed)[-2:]
    for seed in SEEDS
]

positions = np.arange(
    len(SEEDS)
)

ax.set_xticks(
    positions
)

ax.set_yticks(
    positions
)

ax.set_xticklabels(
    seed_labels
)

ax.set_yticklabels(
    seed_labels
)

ax.set_xlabel(
    "Model seed"
)

ax.set_ylabel(
    "Model seed"
)

ax.set_title(
    "Latent neighborhood stability"
)

# Numerical annotations
for i in range(
    len(SEEDS)
):

    for j in range(
        len(SEEDS)
    ):

        value = matrix.iloc[
            i,
            j
        ]

        # White text on dark diagonal
        # black elsewhere
        text_color = (
            "white"
            if value > 0.80
            else "black"
        )

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=8,
            color=text_color,
        )

# Colorbar
cbar = fig.colorbar(
    im,
    ax=ax,
    fraction=0.046,
    pad=0.04,
)


cbar.set_label(
    "Mean kNN overlap"
)

# Clean styling
for spine in ax.spines.values():
    spine.set_visible(False)

ax.tick_params(
    length=0
)

fig.tight_layout()

# Export
fig.savefig(
    OUT_DIR
    / "FigS6A_seed_neighborhood_stability.pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUT_DIR
    / "FigS6A_seed_neighborhood_stability.png",
    dpi=600,
    bbox_inches="tight",
)

plt.show()

# 4. Fig. S6B matched-control calibration

In [ ]:
# ============================================================
# Fig. S6B
# Matched-control calibration of monocyte-program reversion
# ============================================================

MATCHED_DIR = (
    ROOT
    / "Fig6_atlas_program_reversion_Top100_matched"
)

SET_LEVEL_FILE = require_file(
    MATCHED_DIR
    / "Fig6_Top100_program_reversion_set_level.csv"
)

set_df = pd.read_csv(
    SET_LEVEL_FILE
)

required = [
    "disease",
    "program",
    "set_type",
    "control_id",
    "mean_efficacy",
]

missing = [
    c
    for c in required
    if c not in set_df.columns
]

if missing:

    raise ValueError(
        f"Matched-control table missing columns:\n{missing}\n"
        f"Observed columns:\n{set_df.columns.tolist()}"
    )

# 1. Plot
fig, axes = plt.subplots(
    1,
    3,
    figsize=(8.4, 3.7),
    sharey=True,
)

rng = np.random.default_rng(
    20260823
)

y_positions = np.array(
    [2, 1, 0]
)

for ax, disease in zip(
    axes,
    DISEASE_ORDER,
):

    disease_df = set_df.loc[
        set_df["disease"]
        == disease
    ].copy()


    ax.axvline(
        0,
        color="0.75",
        linewidth=1,
        zorder=0,
    )

    for y, program in zip(
        y_positions,
        PROGRAM_ORDER,
    ):

        sub = disease_df.loc[
            disease_df["program"]
            == program
        ]

        control_values = (
            sub.loc[
                sub["set_type"]
                == "matched_control",
                "mean_efficacy",
            ]
            .astype(float)
            .to_numpy()
        )

        observed_values = (
            sub.loc[
                sub["set_type"]
                == "atlas_program",
                "mean_efficacy",
            ]
            .astype(float)
            .to_numpy()
        )

        if len(
            observed_values
        ) != 1:

            raise ValueError(
                f"{disease} / {program}: expected exactly "
                f"one observed atlas-program value, found "
                f"{len(observed_values)}."
            )

        # Small deterministic vertical jitter makes the null
        # distribution visible without a violin/kde
        jitter = rng.normal(
            loc=0,
            scale=0.055,
            size=len(
                control_values
            ),
        )

        ax.scatter(
            control_values,
            y + jitter,
            s=11,
            color="0.70",
            alpha=0.55,
            edgecolors="none",
            zorder=1,
        )

        ax.scatter(
            observed_values[0],
            y,
            s=58,
            marker="D",
            color=PROGRAM_COLORS[
                program
            ],
            edgecolor="black",
            linewidth=0.7,
            zorder=3,
        )

        # Median matched control
        ax.scatter(
            np.median(
                control_values
            ),
            y,
            s=35,
            marker="|",
            color="black",
            linewidth=1.5,
            zorder=2,
        )

    ax.set_title(
        DISEASE_LABELS[
            disease
        ]
    )

    ax.set_xlabel(
        "Risk-state shift / clinical gap"
    )

    ax.grid(
        axis="x",
        linewidth=0.5,
        alpha=0.25,
    )

    ax.spines[
        "top"
    ].set_visible(False)

    ax.spines[
        "right"
    ].set_visible(False)

axes[0].set_yticks(
    y_positions
)

axes[0].set_yticklabels(
    [
        PROGRAM_LABELS[p]
        for p in PROGRAM_ORDER
    ]
)

for ax in axes[1:]:

    ax.tick_params(
        axis="y",
        left=False,
        labelleft=False,
    )

fig.suptitle(
    "Matched-control calibration of monocyte program reversion",
    y=1.02,
)

legend_handles = [

    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="none",
        markersize=5,
        markerfacecolor="0.70",
        markeredgecolor="none",
        label="Matched gene sets",
    ),

    Line2D(
        [0],
        [0],
        marker="D",
        linestyle="none",
        markersize=6,
        markerfacecolor="white",
        markeredgecolor="black",
        label="Atlas program",
    ),
]

fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(
        0.5,
        -0.01,
    ),
    ncol=2,
    frameon=False,
)

fig.tight_layout()

save_panel(
    fig,
    "FigS6B_program_reversion_matched_controls",
)

plt.show()

# 5. Fig. S6C seed-wise K90 stability

In [ ]:
# ============================================================
# Fig. S6C
# Seed-wise K90 stability
# ============================================================

from pathlib import Path
import os

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

from scipy.optimize import curve_fit

# 1. Paths
ROOT = resolve_analysis_path(
    os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai"))
)

OUT_DIR = (
    ROOT
    / "FigS6"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TITRATION_FILE = (
    ROOT
    / "Fig6_reference_reversion_titration"
    / "Fig6_RR_titration_RAW.csv"
)

if not TITRATION_FILE.exists():

    raise FileNotFoundError(
        f"Cannot find:\n{TITRATION_FILE}"
    )

# 2. Fixed disease settings
DISEASE_ORDER = [
    "CAR-T_CRS",
    "COVID19",
    "SLE",
]

DISEASE_LABELS = {
    "CAR-T_CRS": "CAR-T CRS",
    "COVID19": "COVID-19",
    "SLE": "SLE",
}

DISEASE_COLORS = {
    "CAR-T_CRS": "#4C78A8",
    "COVID19": "#F58518",
    "SLE": "#54A24B",
}

# 3. Load saved titration results
tit = pd.read_csv(
    TITRATION_FILE
)

print(
    "Using titration table:"
)

print(
    TITRATION_FILE
)


print(
    "\nColumns:"
)

print(
    tit.columns.tolist()
)


display(
    tit.head()
)

# 4. Validate exact expected structure
required_columns = [
    "perturbation_method",
    "profile",
    "eval_disease",
    "seed",
    "K",
    "control_type",
    "normalized_efficacy",
]

missing = [
    col
    for col in required_columns
    if col not in tit.columns
]

if missing:

    raise ValueError(
        f"Missing required columns:\n{missing}"
    )

# ============================================================
# 5. Select ONLY the disease-specific top-attribution
#    reference-state reversion trajectories
#
# Important:
#
# profile == eval_disease
#
# keeps:
#   CAR-T_CRS features evaluated in CAR-T_CRS
#   COVID19 features evaluated in COVID19
#   SLE features evaluated in SLE
#
# and excludes cross-disease trajectories
#
# control_type == "top_attribution"
#
# removes matched-random controls from the K90 fit
# ============================================================

trajectory_df = tit.loc[

    (
        tit[
            "perturbation_method"
        ]
        == "reference_state_reversion"
    )

    &

    (
        tit[
            "control_type"
        ]
        == "top_attribution"
    )

    &

    (
        tit[
            "profile"
        ]
        == tit[
            "eval_disease"
        ]
    )

    &

    (
        tit[
            "eval_disease"
        ].isin(
            DISEASE_ORDER
        )
    )

].copy()

if len(
    trajectory_df
) == 0:

    raise RuntimeError(
        "No disease-specific top-attribution "
        "trajectories were retained."
    )

print(
    "\nRetained trajectory rows:",
    len(
        trajectory_df
    )
)

print(
    "\nRows by disease:"
)

print(
    trajectory_df[
        "eval_disease"
    ].value_counts()
)

print(
    "\nSeeds:"
)

print(
    sorted(
        trajectory_df[
            "seed"
        ].unique()
    )
)

print(
    "\nK range by disease:"
)

display(
    trajectory_df.groupby(
        "eval_disease"
    )["K"].agg(
        [
            "min",
            "max",
            "nunique",
        ]
    )
)

# ============================================================
# 6. Collapse to one point per disease × seed × K
# ============================================================
trajectory = (

    trajectory_df

    .groupby(
        [
            "eval_disease",
            "seed",
            "K",
        ],
        as_index=False,
    )

    .agg(
        normalized_efficacy=(
            "normalized_efficacy",
            "mean",
        )
    )
)

# ============================================================
# 7. Saturating model
#
# E(K) = A * [1 - exp(-K / tau)]
#
# A:
#   fitted maximal normalized reversion efficacy
#
# tau:
#   characteristic saturation scale
#
# K90:
#   K at which fitted efficacy reaches 90% of A
#
# Since:
#   0.90 = 1 - exp(-K90/tau)
#
# therefore:
#   K90 = tau * ln(10)
# ============================================================

def saturation_model(
    K,
    A,
    tau,
):

    return (
        A
        * (
            1
            -
            np.exp(
                -K
                / tau
            )
        )
    )

# 8. Fit separately for every disease × seed
fit_rows = []


for disease in DISEASE_ORDER:

    disease_df = trajectory.loc[
        trajectory[
            "eval_disease"
        ]
        == disease
    ]


    for seed in sorted(
        disease_df[
            "seed"
        ].unique()
    ):

        sub = (

            disease_df.loc[
                disease_df[
                    "seed"
                ]
                == seed
            ]

            .sort_values(
                "K"
            )

            .dropna(
                subset=[
                    "K",
                    "normalized_efficacy",
                ]
            )
        )


        K = (
            sub[
                "K"
            ]
            .astype(float)
            .to_numpy()
        )


        Y = (
            sub[
                "normalized_efficacy"
            ]
            .astype(float)
            .to_numpy()
        )


        if len(
            np.unique(
                K
            )
        ) < 4:

            print(
                f"Skipping {disease}, seed {seed}: "
                f"too few K points."
            )

            continue

        # Data-driven initial values
        A0 = max(
            float(
                np.nanmax(
                    Y
                )
            ),
            1e-4,
        )

        half_target = (
            0.5
            * A0
        )

        half_idx = int(
            np.nanargmin(
                np.abs(
                    Y
                    - half_target
                )
            )
        )

        tau0 = max(
            float(
                K[
                    half_idx
                ]
            ),
            1.0,
        )

        # Fit
        try:

            popt, _ = curve_fit(

                saturation_model,

                K,
                Y,

                p0=[
                    A0,
                    tau0,
                ],

                bounds=(
                    [
                        0,
                        1e-6,
                    ],
                    [
                        np.inf,
                        np.inf,
                    ],
                ),

                maxfev=50000,
            )


            A_hat = float(
                popt[0]
            )

            tau_hat = float(
                popt[1]
            )


            K90 = (
                tau_hat
                * np.log(
                    10
                )
            )

            fitted = saturation_model(
                K,
                A_hat,
                tau_hat,
            )

            ss_res = np.sum(
                (
                    Y
                    - fitted
                )
                ** 2
            )

            ss_tot = np.sum(
                (
                    Y
                    - np.mean(
                        Y
                    )
                )
                ** 2
            )

            r2 = (
                1
                -
                ss_res
                / ss_tot
                if ss_tot > 0
                else np.nan
            )

            fit_rows.append({

                "disease":
                    disease,

                "seed":
                    int(
                        seed
                    ),

                "A_hat":
                    A_hat,

                "tau_hat":
                    tau_hat,

                "K90":
                    K90,

                "R2":
                    r2,

                "n_K":
                    len(
                        K
                    ),

                "K_max_observed":
                    float(
                        np.max(
                            K
                        )
                    ),
            })


        except Exception as e:

            print(
                f"Fit failed: "
                f"{disease}, seed {seed}\n"
                f"{e}"
            )

# 9. Save fitted K90 values
k90_seed_df = pd.DataFrame(
    fit_rows
)


if len(
    k90_seed_df
) == 0:

    raise RuntimeError(
        "No seed-wise K90 fits succeeded."
    )


k90_seed_df.to_csv(
    OUT_DIR
    / "FigS6C_seedwise_K90.csv",
    index=False,
)


print(
    "\nSeed-wise fitted K90:"
)

display(
    k90_seed_df
)


print(
    "\nK90 summary:"
)

display(

    k90_seed_df

    .groupby(
        "disease"
    )["K90"]

    .agg(
        [
            "median",
            "mean",
            "std",
            "min",
            "max",
        ]
    )
)

# 10. Plot
fig, ax = plt.subplots(
    figsize=(
        4.6,
        3.8,
    )
)


x_positions = {
    disease: i
    for i, disease
    in enumerate(
        DISEASE_ORDER
    )
}

for disease in DISEASE_ORDER:

    sub = (

        k90_seed_df.loc[
            k90_seed_df[
                "disease"
            ]
            == disease
        ]

        .sort_values(
            "seed"
        )
    )

    x0 = x_positions[
        disease
    ]

    # Deterministic small horizontal offsets
    offsets = np.linspace(
        -0.10,
        0.10,
        len(
            sub
        ),
    )

    ax.scatter(

        x0
        + offsets,

        sub[
            "K90"
        ],

        s=46,

        color=DISEASE_COLORS[
            disease
        ],

        edgecolor="black",

        linewidth=0.5,

        zorder=3,
    )

    median_k90 = float(
        sub[
            "K90"
        ].median()
    )

    ax.hlines(

        median_k90,

        x0 - 0.20,
        x0 + 0.20,

        color="black",

        linewidth=1.6,

        zorder=2,
    )

# 11. Styling
ax.set_xticks(
    range(
        len(
            DISEASE_ORDER
        )
    )
)

ax.set_xticklabels(
    [
        DISEASE_LABELS[
            disease
        ]
        for disease
        in DISEASE_ORDER
    ]
)

ax.set_ylabel(
    "K90"
)

ax.set_title(
    "K90 stability across model seeds"
)

ax.grid(
    axis="y",
    linewidth=0.6,
    alpha=0.25,
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()

# 12. Export
fig.savefig(
    OUT_DIR
    / "FigS6C_seedwise_K90.pdf",
    bbox_inches="tight",
)


fig.savefig(
    OUT_DIR
    / "FigS6C_seedwise_K90.png",
    dpi=600,
    bbox_inches="tight",
)


plt.show()

# 6. Fig. S6D expression polarity of disease-specific K90 genes

In [ ]:
# ============================================================
# Fig. S6D
# Expression polarity of disease-specific K90 genes
#
# Expression contrast:
#   patient-balanced mean expression in risk state
#   minus
#   patient-balanced mean expression in reference state
# ============================================================

from pathlib import Path
import os

import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
import matplotlib.pyplot as plt

analysis_start_dir = Path(globals().get("analysis_start_dir", os.getcwd()))

def resolve_analysis_path(path_setting):
    path = Path(path_setting)
    return path if path.is_absolute() else analysis_start_dir / path

# 1. Paths
ROOT = resolve_analysis_path(
    os.environ.get("MAE_WORK_DIR", os.path.join("outputs", "ai"))
)

OUT_DIR = ROOT / "FigS6"

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ADATA_FILE = (
    ROOT
    / "adata_mono_MAE_HVG5000.h5ad"
)

K90_DIR = (
    ROOT
    / "Fig6_reference_reversion_titration"
    / "K90_defined_gene_sets"
)

# 2. Fixed disease definitions
DISEASE_ORDER = [
    "CAR-T_CRS",
    "COVID19",
    "SLE",
]

DISEASE_LABELS = {
    "CAR-T_CRS": "CAR-T CRS",
    "COVID19": "COVID-19",
    "SLE": "SLE",
}

# 3. Load the most recent K90 gene sets for each disease
def find_current_k90_file(disease):

    matches = list(
        K90_DIR.glob(
            f"Fig6_{disease}_K90_Top*_risk_features.csv"
        )
    )

    if len(matches) == 0:

        raise FileNotFoundError(
            f"No K90 file found for {disease}"
        )

    # Extract K from filename and choose the largest/current K
    def extract_k(path):

        stem = path.stem

        text = (
            stem
            .split("_K90_Top")[1]
            .split("_risk_features")[0]
        )

        return int(text)

    matches = sorted(
        matches,
        key=extract_k,
    )

    return matches[-1]

K90_FILES = {
    disease:
        find_current_k90_file(disease)

    for disease
    in DISEASE_ORDER
}

for disease, path in K90_FILES.items():

    print(
        f"{disease:10s} -> {path.name}"
    )

# 4. Read K90 gene membership
k90_genes = {}

for disease, path in K90_FILES.items():

    df = pd.read_csv(
        path
    )

    if "gene" not in df.columns:

        raise ValueError(
            f"'gene' column missing from:\n{path}"
        )

    genes = (
        df["gene"]
        .astype(str)
        .drop_duplicates()
        .tolist()
    )

    k90_genes[
        disease
    ] = genes


    print(
        f"{disease}: "
        f"{len(genes)} K90 genes"
    )

# Union of all disease-specific K90 genes
union_genes = sorted(
    set().union(
        *[
            set(v)
            for v in k90_genes.values()
        ]
    )
)

print(
    "\nUnion of K90 genes:",
    len(union_genes)
)

# 5. Open frozen 5,000-HVG AnnData in BACKED mode
# backed='r' prevents loading the full ~306k × 5000 matrix into RAM
adata = ad.read_h5ad(
    ADATA_FILE,
    backed="r",
)

print(
    "\nAnnData:",
    adata.shape
)

required_obs = [
    "disease",
    "patient_id",
    "severity",
    "sampling_time",
]

missing_obs = [
    col
    for col in required_obs
    if col not in adata.obs.columns
]

if missing_obs:

    raise ValueError(
        f"Missing required metadata columns:\n"
        f"{missing_obs}\n\n"
        f"Available obs columns:\n"
        f"{adata.obs.columns.tolist()}"
    )

# 6. Verify all current K90 genes are in the frozen HVG space
available_genes = set(
    adata.var_names.astype(str)
)

missing_genes = [
    gene
    for gene in union_genes
    if gene not in available_genes
]

if missing_genes:

    raise ValueError(
        f"{len(missing_genes)} K90 genes are absent "
        f"from the frozen 5,000-HVG AnnData.\n"
        f"First missing genes:\n"
        f"{missing_genes[:20]}"
    )

# 7. Load ONLY the union of K90 genes into memory
gene_indices = np.array(
    [
        adata.var_names.get_loc(
            gene
        )
        for gene in union_genes
    ],
    dtype=int,
)


# Backed sparse slicing
X = adata.X[
    :,
    gene_indices
]

# Convert to an ordinary in-memory sparse matrix
if not sp.issparse(X):

    X = sp.csr_matrix(
        X
    )

else:

    X = X.tocsr()

obs = adata.obs[
    required_obs
].copy()

adata.file.close()

print(
    "\nSubset expression matrix:",
    X.shape
)

# 8. Define frozen risk/reference groups
def clinical_masks(obs, disease):

    if disease == "CAR-T_CRS":

        risk = (
            (obs["disease"] == "CAR-T_CRS")
            &
            (obs["severity"] == "Severe")
            &
            (
                obs["sampling_time"]
                == "CAR-T_CRS_pro"
            )
        )

        reference = (
            (obs["disease"] == "CAR-T_CRS")
            &
            (obs["severity"] == "Severe")
            &
            (
                obs["sampling_time"].isin(
                    [
                        "CAR-T_CRS_before",
                        "CAR-T_CRS_con",
                    ]
                )
            )
        )

    elif disease == "COVID19":

        risk = (
            (obs["disease"] == "COVID19")
            &
            (obs["severity"] == "Severe")
            &
            (
                obs["sampling_time"]
                == "COVID19_pro"
            )
        )

        reference = (
            (obs["disease"] == "COVID19")
            &
            (
                (
                    (obs["severity"] == "Moderate")
                    &
                    (
                        obs["sampling_time"]
                        == "COVID19_pro"
                    )
                )
                |
                (
                    (obs["severity"] == "Severe")
                    &
                    (
                        obs["sampling_time"]
                        == "COVID19_con"
                    )
                )
            )
        )

    elif disease == "SLE":

        risk = (
            (obs["disease"] == "SLE")
            &
            (obs["severity"] == "Severe")
        )

        reference = (
            (obs["disease"] == "SLE")
            &
            (obs["severity"] == "Moderate")
        )

    else:

        raise ValueError(
            disease
        )

    return (
        risk.to_numpy(),
        reference.to_numpy(),
    )

# 9. Patient-balanced mean expression
#
# For each clinical state:
#
#   cells
#      ↓
#   mean expression per patient
#      ↓
#   equal-weight mean across patients
#
# This prevents patients with many monocytes from dominating
# the expression contrast

def patient_balanced_expression(
    X,
    obs,
    mask,
):

    selected_idx = np.flatnonzero(
        mask
    )


    if len(
        selected_idx
    ) == 0:

        raise ValueError(
            "Clinical mask selected zero cells."
        )


    selected_obs = obs.iloc[
        selected_idx
    ]


    patient_ids = (
        selected_obs[
            "patient_id"
        ]
        .astype(str)
        .unique()
    )

    patient_means = []

    for patient in patient_ids:

        patient_local_mask = (
            selected_obs[
                "patient_id"
            ].astype(str)
            == patient
        ).to_numpy()


        patient_global_idx = (
            selected_idx[
                patient_local_mask
            ]
        )

        patient_mean = np.asarray(
            X[
                patient_global_idx,
                :
            ].mean(
                axis=0
            )
        ).ravel()

        patient_means.append(
            patient_mean
        )

    patient_means = np.vstack(
        patient_means
    )

    balanced_mean = np.mean(
        patient_means,
        axis=0,
    )

    return (
        balanced_mean,
        len(patient_ids),
        len(selected_idx),
    )

# 10. Calculate disease-specific expression shifts
expression_rows = []

for disease in DISEASE_ORDER:

    risk_mask, ref_mask = (
        clinical_masks(
            obs,
            disease,
        )
    )

    (
        risk_mean,
        n_risk_patients,
        n_risk_cells,
    ) = patient_balanced_expression(
        X,
        obs,
        risk_mask,
    )

    (
        ref_mean,
        n_ref_patients,
        n_ref_cells,
    ) = patient_balanced_expression(
        X,
        obs,
        ref_mask,
    )

    shift = (
        risk_mean
        - ref_mean
    )

    print(
        f"\n{disease}"
    )

    print(
        f"  Risk: "
        f"{n_risk_patients} patients, "
        f"{n_risk_cells:,} monocytes"
    )

    print(
        f"  Reference: "
        f"{n_ref_patients} patients, "
        f"{n_ref_cells:,} monocytes"
    )

    for gene, risk_value, ref_value, delta in zip(
        union_genes,
        risk_mean,
        ref_mean,
        shift,
    ):

        expression_rows.append({

            "gene":
                gene,

            "disease":
                disease,

            "risk_mean_expression":
                float(
                    risk_value
                ),

            "reference_mean_expression":
                float(
                    ref_value
                ),

            "risk_state_expression_shift":
                float(
                    delta
                ),

            "n_risk_patients":
                n_risk_patients,

            "n_reference_patients":
                n_ref_patients,

            "n_risk_cells":
                n_risk_cells,

            "n_reference_cells":
                n_ref_cells,
        })

expression_df = pd.DataFrame(
    expression_rows
)

# Save the reusable source table
EXPRESSION_TABLE = (
    OUT_DIR
    / "FigS6_K90_risk_reference_expression_ALL.csv"
)

expression_df.to_csv(
    EXPRESSION_TABLE,
    index=False,
)

print(
    "\nSaved reusable expression table:"
)

print(
    EXPRESSION_TABLE
)

# 11. Restrict each disease to its CURRENT K90 set
plot_tables = {}

for disease in DISEASE_ORDER:

    genes = set(
        k90_genes[
            disease
        ]
    )

    plot_tables[
        disease
    ] = expression_df.loc[
        (
            expression_df[
                "disease"
            ]
            == disease
        )
        &
        (
            expression_df[
                "gene"
            ].isin(
                genes
            )
        )
    ].copy()


    print(
        f"{disease}: "
        f"{len(plot_tables[disease])} plotted genes"
    )

# 12. Plot
N_LABEL_POSITIVE = 8
N_LABEL_NEGATIVE = 5

fig, ax = plt.subplots(
    figsize=(8.8, 5.2)
)

x_centers = {
    "CAR-T_CRS": 0,
    "COVID19": 1,
    "SLE": 2,
}

rng = np.random.default_rng(
    20260823
)

for disease in DISEASE_ORDER:

    df = plot_tables[
        disease
    ]

    x0 = x_centers[
        disease
    ]

    y = df[
        "risk_state_expression_shift"
    ].to_numpy()

    jitter = rng.uniform(
        -0.27,
        0.27,
        size=len(df),
    )

    x = (
        x0
        + jitter
    )

    # Background K90 genes
    ax.scatter(
        x,
        y,
        s=26,
        color="0.72",
        alpha=0.65,
        edgecolors="none",
        zorder=1,
    )

    # Label strongest positive and negative shifts
    positive = df.nlargest(
        N_LABEL_POSITIVE,
        "risk_state_expression_shift",
    )

    negative = df.nsmallest(
        N_LABEL_NEGATIVE,
        "risk_state_expression_shift",
    )

    highlight = pd.concat(
        [
            positive,
            negative,
        ]
    ).drop_duplicates(
        "gene"
    )

    gene_to_x = dict(
        zip(
            df["gene"],
            x,
        )
    )

    for _, row in highlight.iterrows():

        gene = row[
            "gene"
        ]

        value = float(
            row[
                "risk_state_expression_shift"
            ]
        )

        x_gene = gene_to_x[
            gene
        ]

        point_color = (
            "#C52F2F"
            if value > 0
            else "#3E83B8"
        )

        ax.scatter(
            x_gene,
            value,
            s=54,
            color=point_color,
            edgecolor="none",
            zorder=3,
        )

        ax.annotate(
            gene,
            (
                x_gene,
                value,
            ),
            xytext=(
                3,
                4
                if value >= 0
                else -11,
            ),
            textcoords="offset points",
            fontsize=7,
            ha="left",
        )

# 13. Styling
ax.axhline(
    0,
    color="black",
    linewidth=1,
)

ax.set_xticks(
    [
        0,
        1,
        2,
    ]
)

ax.set_xticklabels(
    [
        "CAR-T CRS",
        "COVID-19",
        "SLE",
    ]
)

ax.set_ylabel(
    "Risk-state expression shift"
)

ax.set_title(
    "Expression polarity of disease-specific K90 genes"
)

ax.spines[
    "top"
].set_visible(False)

ax.spines[
    "right"
].set_visible(False)

fig.tight_layout()

# 14. Export
fig.savefig(
    OUT_DIR
    / "FigS6D_K90_gene_expression_profiles.pdf",
    bbox_inches="tight",
)

fig.savefig(
    OUT_DIR
    / "FigS6D_K90_gene_expression_profiles.png",
    dpi=600,
    bbox_inches="tight",
)

plt.show()